# 03_Retina Super-Resolution — SRGAN (Google Colab)
망막 OCT 영상(grayscale 1채널)을 대상으로 SRGAN을 훈련합니다.

**주요 수정 사항**
- 입력/출력 채널: 3 → **1** (grayscale OCT)
- VGG Perceptual loss: 1채널을 3채널로 복제 후 VGG19 통과
- Upscale factor: `SCALE` 변수로 ×2 / ×4 선택 가능
- GAN 안정화: `D_UPDATE_FREQ`, learning rate, loss 가중치 조정 지원

In [ ]:
!pip install -q scikit-image

In [ ]:
# ── Google Drive 마운트 ──
# 이미 마운트된 경우 자동 스킵됩니다.
# 'Transport endpoint is not connected' 오류 발생 시:
#   force_remount=True 로 변경 후 재실행하세요.
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


In [ ]:
import os, glob, math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from tensorflow.keras.layers import (
    Add, BatchNormalization, Conv2D, Dense, Flatten,
    Input, LeakyReLU, PReLU, Lambda
)
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.optimizers import Adam
from skimage.metrics import peak_signal_noise_ratio as calc_psnr
from skimage.metrics import structural_similarity as calc_ssim

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── 경로 및 하이퍼파라미터 ──────────────────────────────────────────────
# Google Drive에 데이터 업로드 후 BASE_DRIVE 경로를 수정하세요.
# 예: Drive 최상위 > 'Retina_SR' 폴더 생성 후 data/ 폴더를 그 안에 업로드

BASE_DRIVE = '/content/drive/MyDrive/Retina_SR/data'  # <-- set your actual path

HR_DIRS = [
    os.path.join(BASE_DRIVE, 'High_Res', 'group1'),
    os.path.join(BASE_DRIVE, 'High_Res', 'group2'),
    os.path.join(BASE_DRIVE, 'High_Res', 'group3'),
]
LR_DIRS = [
    os.path.join(BASE_DRIVE, 'Low_Res', 'group1'),
    os.path.join(BASE_DRIVE, 'Low_Res', 'group2'),
    os.path.join(BASE_DRIVE, 'Low_Res', 'group3'),
]

SCALE          = 4      # upscale factor: 4 (change to 2 if LR/HR ratio is 1:2)
HR_CROP        = 128    # HR patch size (reduce if OOM)
LR_CROP        = HR_CROP // SCALE
NUM_FILTERS    = 64
NUM_RES_BLOCKS = 16
BATCH_SIZE     = 8
PRETRAIN_STEPS = 1000
SRGAN_STEPS    = 2000
LR_G           = 1e-4
LR_D           = 1e-4
D_UPDATE_FREQ  = 1      # D update steps per G update
W_PIXEL        = 1.0
W_PERCEPTUAL   = 0.006
W_ADV          = 0.001

SAVE_DIR = '/content/srgan_results'
os.makedirs(os.path.join(SAVE_DIR, 'checkpoints'), exist_ok=True)
print(f'LR_CROP={LR_CROP}, HR_CROP={HR_CROP}, SCALE={SCALE}x')

In [ ]:
# ── 데이터 경로 수집 및 스케일 확인 ──────────────────────────────────────
IMG_EXTS = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.tiff')

def collect_paths(dirs):
    paths = []
    for d in dirs:
        for ext in IMG_EXTS:
            paths.extend(sorted(glob.glob(os.path.join(d, ext))))
    return paths

hr_paths = collect_paths(HR_DIRS)
lr_paths = collect_paths(LR_DIRS)
print(f'HR images: {len(hr_paths)}, LR images: {len(lr_paths)}')
assert len(hr_paths) == len(lr_paths) > 0, 'No images found or counts mismatch — check paths'

s_hr = np.array(Image.open(hr_paths[0]).convert('L'))
s_lr = np.array(Image.open(lr_paths[0]).convert('L'))
print(f'HR shape: {s_hr.shape}, LR shape: {s_lr.shape}')
actual_scale = s_hr.shape[0] // s_lr.shape[0]
print(f'Detected scale: {actual_scale}x  (config: SCALE={SCALE})')
if actual_scale != SCALE:
    print(f'[WARNING] Set SCALE={actual_scale} and re-run the config cell!')

n_total = len(hr_paths)
idx = np.random.permutation(n_total)
n_train = int(n_total * 0.8)
tr, va = idx[:n_train], idx[n_train:]
hr_train, lr_train = [hr_paths[i] for i in tr], [lr_paths[i] for i in tr]
hr_val,   lr_val   = [hr_paths[i] for i in va], [lr_paths[i] for i in va]
print(f'Train: {len(hr_train)}, Val: {len(hr_val)}')

In [ ]:
# ── tf.data 파이프라인 ────────────────────────────────────────────────────

# ── Drive 재연결 확인 ──
# Drive가 끊어진 경우 아래 주석을 해제하고 실행하세요.
# from google.colab import drive; drive.mount('/content/drive', force_remount=True)

def load_pair(hr_path, lr_path):
    def _read(p):
        img = tf.io.read_file(p)
        img = tf.image.decode_image(img, channels=1, expand_animations=False)
        return tf.cast(img, tf.float32) / 255.0
    hr = _read(hr_path)
    lr = _read(lr_path)

    hr_h, hr_w = tf.shape(hr)[0], tf.shape(hr)[1]
    lr_h, lr_w = tf.shape(lr)[0], tf.shape(lr)[1]

    # HR 랜덤 크롭
    top  = tf.random.uniform((), 0, tf.maximum(hr_h - HR_CROP + 1, 1), dtype=tf.int32)
    left = tf.random.uniform((), 0, tf.maximum(hr_w - HR_CROP + 1, 1), dtype=tf.int32)
    hr = hr[top:top+HR_CROP, left:left+HR_CROP, :]
    hr = tf.image.resize_with_crop_or_pad(hr, HR_CROP, HR_CROP)  # ensure fixed size

    # 대응 LR 크롭 — 경계 초과 방지
    lt = tf.minimum(top  // SCALE, tf.maximum(lr_h - LR_CROP, 0))
    ll = tf.minimum(left // SCALE, tf.maximum(lr_w - LR_CROP, 0))
    lr = lr[lt:lt+LR_CROP, ll:ll+LR_CROP, :]
    lr = tf.image.resize_with_crop_or_pad(lr, LR_CROP, LR_CROP)  # ensure fixed size

    # 좌우 반전 augmentation
    flip = tf.random.uniform(()) > 0.5
    hr = tf.cond(flip, lambda: tf.image.flip_left_right(hr), lambda: hr)
    lr = tf.cond(flip, lambda: tf.image.flip_left_right(lr), lambda: lr)
    return lr, hr


def make_ds(hr_list, lr_list, bs, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((hr_list, lr_list))
    if shuffle: ds = ds.shuffle(len(hr_list))
    return (ds
            .map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(bs)
            .prefetch(tf.data.AUTOTUNE))


train_ds = make_ds(hr_train, lr_train, BATCH_SIZE)
val_ds   = make_ds(hr_val,   lr_val,   BATCH_SIZE, shuffle=False)

# 샘플 1장으로 파이프라인 동작 확인
print('Testing data pipeline...')
for lr_b, hr_b in val_ds.take(1):
    print(f'LR batch shape: {lr_b.shape}, HR batch shape: {hr_b.shape}')
    n_show = min(4, lr_b.shape[0])
    fig, axes = plt.subplots(2, n_show, figsize=(3*n_show, 6))
    for i in range(n_show):
        axes[0,i].imshow(lr_b[i,...,0], cmap='gray'); axes[0,i].axis('off')
        axes[0,i].set_title(f'LR {lr_b.shape[1]}x{lr_b.shape[2]}')
        axes[1,i].imshow(hr_b[i,...,0], cmap='gray'); axes[1,i].axis('off')
        axes[1,i].set_title(f'HR {hr_b.shape[1]}x{hr_b.shape[2]}')
    plt.suptitle('Data Samples (LR top / HR bottom)'); plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'data_sample.png'), dpi=120); plt.show()
print('Pipeline OK')


In [ ]:
# ── SRGAN 모델 정의 (grayscale 1채널 적용) ───────────────────────────────
def norm_m11(x):     return x * 2.0 - 1.0
def denorm_m11(x):   return (x + 1.0) / 2.0
def pix_shuffle(s):  return lambda x: tf.nn.depth_to_space(x, s)

def upsample_block(x_in, nf):
    x = Conv2D(nf, 3, padding='same')(x_in)
    x = Lambda(pix_shuffle(2))(x)
    return PReLU(shared_axes=[1,2])(x)

def res_block(x_in, nf, mom=0.8):
    x = Conv2D(nf, 3, padding='same')(x_in)
    x = BatchNormalization(momentum=mom)(x)
    x = PReLU(shared_axes=[1,2])(x)
    x = Conv2D(nf, 3, padding='same')(x)
    x = BatchNormalization(momentum=mom)(x)
    return Add()([x_in, x])

def build_generator(nf=64, n_res=16, scale=4):
    x_in = Input(shape=(None, None, 1))         # grayscale 1-channel input
    x = Lambda(norm_m11)(x_in)
    x = Conv2D(nf, 9, padding='same')(x)
    x = skip = PReLU(shared_axes=[1,2])(x)
    for _ in range(n_res): x = res_block(x, nf)
    x = Conv2D(nf, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Add()([skip, x])
    for _ in range(int(math.log2(scale))): x = upsample_block(x, nf*4)
    x = Conv2D(1, 9, padding='same', activation='tanh')(x)  # 1-channel output
    x = Lambda(denorm_m11)(x)
    return Model(x_in, x, name='generator')

def disc_block(x_in, nf, s=1, bn=True, mom=0.8):
    x = Conv2D(nf, 3, strides=s, padding='same')(x_in)
    if bn: x = BatchNormalization(momentum=mom)(x)
    return LeakyReLU(0.2)(x)

def build_discriminator(hr_size, nf=64):
    x_in = Input(shape=(hr_size, hr_size, 1))   # grayscale 1-channel
    x = Lambda(norm_m11)(x_in)
    x = disc_block(x, nf,    bn=False)
    x = disc_block(x, nf,    s=2)
    x = disc_block(x, nf*2)
    x = disc_block(x, nf*2,  s=2)
    x = disc_block(x, nf*4)
    x = disc_block(x, nf*4,  s=2)
    x = disc_block(x, nf*8)
    x = disc_block(x, nf*8,  s=2)
    x = Flatten()(x)
    x = Dense(1024)(x)
    x = LeakyReLU(0.2)(x)
    x = Dense(1, activation='sigmoid')(x)
    return Model(x_in, x, name='discriminator')

def build_vgg(layer_idx=20):
    """grayscale (1ch) → replicate to 3ch → VGG19 feature extraction"""
    vgg = VGG19(input_shape=(None,None,3), include_top=False, weights='imagenet')
    vgg.trainable = False
    feat = Model(vgg.input, vgg.layers[layer_idx].output)
    inp = Input(shape=(None, None, 1))
    x = Lambda(lambda t: tf.repeat(t * 255.0, 3, axis=-1))(inp)
    return Model(inp, feat(x), name='vgg_extractor')

generator     = build_generator(NUM_FILTERS, NUM_RES_BLOCKS, SCALE)
discriminator = build_discriminator(HR_CROP, NUM_FILTERS)
vgg_extractor = build_vgg(layer_idx=20)

print(f'Generator     params: {generator.count_params():,}')
print(f'Discriminator params: {discriminator.count_params():,}')
generator.summary(line_length=80)

In [ ]:
# ── 손실 함수 & 옵티마이저 ────────────────────────────────────────────────
opt_g = Adam(LR_G, beta_1=0.9)
opt_d = Adam(LR_D, beta_1=0.9)
bce   = tf.keras.losses.BinaryCrossentropy()
mse   = tf.keras.losses.MeanSquaredError()

@tf.function
def l_pixel(hr, sr):       return mse(hr, sr)

@tf.function
def l_perceptual(hr, sr):
    return mse(vgg_extractor(hr, training=False),
               vgg_extractor(sr, training=False))

@tf.function
def l_adv_g(sr):
    p = discriminator(sr, training=False)
    return bce(tf.ones_like(p), p)

@tf.function
def l_disc(hr, sr):
    r = discriminator(hr, training=True)
    f = discriminator(sr, training=True)
    return 0.5 * (bce(tf.ones_like(r), r) + bce(tf.zeros_like(f), f))


@tf.function
def pretrain_step(lr_b, hr_b):
    with tf.GradientTape() as t:
        sr   = generator(lr_b, training=True)
        loss = l_pixel(hr_b, sr)
    opt_g.apply_gradients(zip(t.gradient(loss, generator.trainable_variables),
                               generator.trainable_variables))
    return loss


@tf.function
def srgan_g_step(lr_b, hr_b):
    with tf.GradientTape() as t:
        sr      = generator(lr_b, training=True)
        lp      = l_pixel(hr_b, sr)
        lperc   = l_perceptual(hr_b, sr)
        ladv    = l_adv_g(sr)
        total   = W_PIXEL*lp + W_PERCEPTUAL*lperc + W_ADV*ladv
    opt_g.apply_gradients(zip(t.gradient(total, generator.trainable_variables),
                               generator.trainable_variables))
    return total, lp, lperc, ladv


@tf.function
def srgan_d_step(lr_b, hr_b):
    sr = generator(lr_b, training=False)
    with tf.GradientTape() as t:
        loss = l_disc(hr_b, sr)
    opt_d.apply_gradients(zip(t.gradient(loss, discriminator.trainable_variables),
                               discriminator.trainable_variables))
    return loss

In [ ]:
# ── Phase 1: Generator 사전 훈련 (MSE only) ──────────────────────────────
print('=== Phase 1: Pre-training Generator with MSE ===')
pre_losses = []
it = iter(train_ds.repeat())
for step in range(1, PRETRAIN_STEPS+1):
    lr_b, hr_b = next(it)
    loss = pretrain_step(lr_b, hr_b)
    pre_losses.append(float(loss))
    if step % 100 == 0 or step == 1:
        print(f'  [{step:4d}/{PRETRAIN_STEPS}]  MSE: {loss:.6f}')

plt.figure(figsize=(8,3))
plt.plot(pre_losses); plt.title('Pre-train Loss (MSE)')
plt.xlabel('Step'); plt.ylabel('MSE'); plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'pretrain_loss.png'), dpi=120); plt.show()
generator.save_weights(os.path.join(SAVE_DIR, 'checkpoints', 'gen_pretrained.weights.h5'))
print('Saved pretrained weights.')

In [ ]:
# ── Phase 2: SRGAN 훈련 ──────────────────────────────────────────────────
print('=== Phase 2: SRGAN Training (pixel + perceptual + adversarial) ===')
log = {k: [] for k in ('g_total','g_pix','g_perc','g_adv','d_loss')}
it = iter(train_ds.repeat())

for step in range(1, SRGAN_STEPS+1):
    d_loss = 0.0
    for _ in range(D_UPDATE_FREQ):
        lr_b, hr_b = next(it)
        d_loss = float(srgan_d_step(lr_b, hr_b))
    lr_b, hr_b = next(it)
    g_tot, g_pix, g_perc, g_adv = srgan_g_step(lr_b, hr_b)
    for k, v in zip(('g_total','g_pix','g_perc','g_adv','d_loss'),
                    (g_tot, g_pix, g_perc, g_adv, d_loss)):
        log[k].append(float(v))
    if step % 200 == 0 or step == 1:
        print(f'  [{step:4d}/{SRGAN_STEPS}]  G={g_tot:.4f}  '
              f'pix={g_pix:.4f}  perc={g_perc:.4f}  adv={g_adv:.4f}  D={d_loss:.4f}')

generator.save_weights(os.path.join(SAVE_DIR, 'checkpoints', 'gen_srgan.weights.h5'))
discriminator.save_weights(os.path.join(SAVE_DIR, 'checkpoints', 'disc_srgan.weights.h5'))
print('Saved SRGAN weights.')

In [ ]:
# ── 학습 곡선 ──────────────────────────────────────────────────────────────
steps = range(1, len(log['g_total'])+1)
fig, axes = plt.subplots(1, 3, figsize=(16,4))
axes[0].plot(steps, log['g_total']);  axes[0].set_title('G Total Loss'); axes[0].set_xlabel('Step')
axes[1].plot(steps, log['g_pix'],   label=f'Pixel x{W_PIXEL}')
axes[1].plot(steps, log['g_perc'],  label=f'Perceptual x{W_PERCEPTUAL}')
axes[1].plot(steps, log['g_adv'],   label=f'Adversarial x{W_ADV}')
axes[1].set_title('G Loss Components'); axes[1].set_xlabel('Step'); axes[1].legend()
axes[2].plot(steps, log['d_loss'], color='orange'); axes[2].set_title('D Loss'); axes[2].set_xlabel('Step')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# ── 정량 평가 (PSNR / SSIM) ────────────────────────────────────────────────
def evaluate(ds, n_batches=10):
    psnr_lr, psnr_sr, ssim_lr, ssim_sr = [], [], [], []
    for i, (lr_b, hr_b) in enumerate(ds):
        if i >= n_batches: break
        sr_b = np.clip(generator(lr_b, training=False).numpy(), 0, 1)
        for j in range(lr_b.shape[0]):
            hr   = hr_b[j,...,0].numpy()
            sr   = sr_b[j,...,0]
            lr_u = np.array(Image.fromarray(
                       (lr_b[j,...,0].numpy()*255).astype(np.uint8)
                   ).resize((HR_CROP,HR_CROP), Image.BICUBIC)).astype(np.float32)/255.0
            psnr_lr.append(calc_psnr(hr, lr_u, data_range=1.0))
            psnr_sr.append(calc_psnr(hr, sr,   data_range=1.0))
            ssim_lr.append(calc_ssim(hr, lr_u,  data_range=1.0))
            ssim_sr.append(calc_ssim(hr, sr,    data_range=1.0))
    return {'PSNR Bicubic': np.mean(psnr_lr), 'PSNR SRGAN': np.mean(psnr_sr),
            'SSIM Bicubic': np.mean(ssim_lr), 'SSIM SRGAN': np.mean(ssim_sr)}

metrics = evaluate(val_ds)
print('=== Validation Set Metrics ===')
for k, v in metrics.items(): print(f'  {k:20s}: {v:.4f}')

In [ ]:
# ── 대표 결과 이미지 시각화 ─────────────────────────────────────────────────
def show_results(ds, n=4, save_path=None):
    for lr_b, hr_b in ds.take(1):
        sr_b = np.clip(generator(lr_b, training=False).numpy(), 0, 1)
        lr_b, hr_b = lr_b.numpy(), hr_b.numpy()
        break
    n = min(n, lr_b.shape[0])
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    if n == 1: axes = axes[None]
    for i in range(n):
        hr   = hr_b[i,...,0]
        sr   = sr_b[i,...,0]
        lr_u = np.array(Image.fromarray(
                   (lr_b[i,...,0]*255).astype(np.uint8)
               ).resize((HR_CROP,HR_CROP), Image.BICUBIC)).astype(np.float32)/255.0
        p_lr = calc_psnr(hr, lr_u, data_range=1.0)
        p_sr = calc_psnr(hr, sr,   data_range=1.0)
        s_lr = calc_ssim(hr, lr_u,  data_range=1.0)
        s_sr = calc_ssim(hr, sr,    data_range=1.0)
        titles = [f'LR (Bicubic)\nPSNR={p_lr:.2f} SSIM={s_lr:.4f}',
                  f'SR (SRGAN)\nPSNR={p_sr:.2f} SSIM={s_sr:.4f}',
                  'HR (Ground Truth)']
        for ax, img, title in zip(axes[i], [lr_u, sr, hr], titles):
            ax.imshow(img, cmap='gray', vmin=0, vmax=1)
            ax.set_title(title, fontsize=9); ax.axis('off')
    plt.suptitle('LR (Bicubic) | SR (SRGAN) | HR (GT)', fontsize=12)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()

show_results(val_ds, n=4,
             save_path=os.path.join(SAVE_DIR, 'result_comparison.png'))
print(f'Saved: {SAVE_DIR}/result_comparison.png')

In [ ]:
# ── GAN 훈련 안정화 가이드 ───────────────────────────────────────────────
# D loss ≈ 0 (D 완승): LR_D 줄이기 또는 D_UPDATE_FREQ = 0
# G adv loss 폭발 : W_ADV = 0.0005 로 줄이기
# 품질 개선 없음   : W_PERCEPTUAL = 0.01, PRETRAIN_STEPS 늘리기
#
# LR 감쇠 스케줄 예시:
# sched = tf.keras.optimizers.schedules.ExponentialDecay(
#     1e-4, decay_steps=500, decay_rate=0.5, staircase=True)
# opt_g = Adam(sched)
#
# Label smoothing (D 안정화):
# real_labels → 0.9,  fake_labels → 0.1
print('GAN stability guide: see comments above')

In [ ]:
# ── 결과물 Google Drive에 저장 ───────────────────────────────────────────
import shutil
dst = os.path.normpath(os.path.join(BASE_DRIVE, '..', 'srgan_results'))
if os.path.exists(dst): shutil.rmtree(dst)
shutil.copytree(SAVE_DIR, dst)
print(f'Saved to Drive: {dst}')
for f in sorted(os.listdir(dst)): print(' ', f)